# PIGEON-AI: H3 grid and toilet assignment

Inspect resolution-9 hexagons and the historical placement proxy. H3 selects cells by their centres; cells remain complete and can cross the SA1 study boundary. Inputs: final features; the interim study area for the rebuild check. Outputs: checks and maps. Grid generation and assignment live in `src/pigeon_ai/geography.py`.

In [ ]:
from pathlib import Path
import sys

# Works when the kernel starts in the project root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pigeon_ai.paths import INTERIM, PROCESSED, CRS

assert (PROCESSED / "sydney_hex_features.gpkg").exists(), "Run python scripts/run_pipeline.py first."


In [ ]:
grid = gpd.read_file(PROCESSED / "sydney_hex_features.gpkg")
assert grid.crs.to_epsg() == 7856
assert grid.hex_id.is_unique
print(f"Hexagons: {len(grid):,}")
print(f"Toilet records: {grid.toilet_count.sum():,}")
print(f"Occupied hexagons: {(grid.toilet_count > 0).sum():,}")
display(grid[["toilet_count"]].describe())

## Boundary semantics

National toilet coordinates are assigned directly with H3 resolution 9, then joined to every selected cell. Zero-count cells remain in the prediction grid. This differs intentionally from clipping toilet points to the SA1 union. Missing coordinates are excluded and reported by the pipeline.

In [ ]:
from pigeon_ai.geography import build_grid

study_path = INTERIM / "boundaries/sydney_study_sa1_final.geojson"
if study_path.exists():
    study = gpd.read_file(study_path)
    rebuilt = build_grid(study)
    assert set(rebuilt.hex_id) == set(grid.hex_id)
    print("Regenerated H3 membership matches exactly.")
else:
    print("Skipped: run python scripts/run_pipeline.py to create the study area.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
grid.plot(ax=ax, column="toilet_count", cmap="YlOrRd", legend=True, linewidth=0)
ax.set(title="Mapped toilet facilities per complete H3 cell")
ax.set_axis_off()
plt.show()